In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
SRC = "/content/drive/MyDrive/tagging/test.h5"
# 目标路径 (Colab 临时盘)
DST = "/content/test.h5"

!rsync -ah --info=progress2 "$SRC" "$DST"

          8.16G 100%   75.70MB/s    0:01:42 (xfr#1, to-chk=0/1)


In [ ]:
import os
import sys
import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split


ROOT_DIR = os.getcwd()
DATA_DIR = ROOT_DIR
THIS_DIR = f'{ROOT_DIR}/drive/MyDrive/tagging/pure_unsmear/joint_final'
MODULE_DIR = THIS_DIR
TAGGING_DIR = os.path.abspath(os.path.join(MODULE_DIR, '..'))

sys.path.insert(0, MODULE_DIR)

import tool  # noqa: E402
import model as model_mod  # noqa: E402
import importlib  # noqa: E402
importlib.reload(tool)  # noqa: E402
importlib.reload(model_mod)  # noqa: E402
from model import ParticleTransformerKD, SharedEncoderUnsmearClassifier  # noqa: E402

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

RUN_NAME = 'unsmear_transformer_sharedencoder_delta_gate_joint_repeat3'
OUT_DIR = os.path.join(MODULE_DIR, 'runs', RUN_NAME)
FIG_DIR = os.path.join(OUT_DIR, 'figs')
CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')
METRICS_DIR = os.path.join(OUT_DIR, 'metrics')
REPEAT_DIR = os.path.join(OUT_DIR, 'repeats')
TABLE_DIR = os.path.join(OUT_DIR, 'tables')

tool.ensure_dir(FIG_DIR)
tool.ensure_dir(CKPT_DIR)
tool.ensure_dir(METRICS_DIR)
tool.ensure_dir(REPEAT_DIR)
tool.ensure_dir(TABLE_DIR)

CONFIG = {
    'data_path': os.path.join(DATA_DIR, 'test.h5'),
    'n_jets': 200000,
    'max_particles': 100,
    'feature_kind': '7d',
    'repeat_seeds': [42, 52, 62],
    'load_shared_baselines': False,
    'load_joint_model': False,
    # 共享分类 backbone 配置：baseline / joint 都从这里派生，保证严格一致。
    'shared_backbone': {
        'input_dim': 7,
        'embed_dim': 128,
        'num_heads': 8,
        'num_layers': 6,
        'ff_dim': 512,
        'dropout': 0.1,
        'add_mask_channel': False,
        'use_positional_embedding': False,
        'max_seq_len': 128,
        'mask_encoder_input': False,
    },
    # joint 初始化控制：可分别为 no_kd / with_kd 选择 backbone 来源，并决定是否冻结到 z；'none' 表示完全不加载 baseline backbone。
    'joint_init': {
        'load_backbone_from_no_kd': 'hlt',
        'load_backbone_from_with_kd': 'hlt+kd',
        'freeze_backbone_to_z': False,
    },
    # 纯分类 baseline 配置会在下面由 shared_backbone 自动生成。
    'tagger': {},
    # joint 专有配置：共享 backbone 参数同样在下面自动并入。
    'joint_model': {
        'unsmear_decoder_layers': 2,
        'unsmear_decoder_heads': 8,
        'unsmear_decoder_ff_dim': 512,
        'unsmear_decoder_dropout': 0.1,
        'return_reco': True,
        'mask_output': True,
        'cls_use_delta_fusion': True,
        'cls_detach_delta_for_cls': False,  # 分类 loss 会通过 fusion 回传到 regression 分支，进行端到端联合训练。
        'cls_gate_hidden_dim': 128,
        'cls_gate_init_bias': -2.0,
        'cls_alpha_init': 0.05,
    },
    'hlt_effects': {
        'pt_threshold_offline': 0,
        'pt_threshold_hlt': 0,
        'pt_resolution': 0.10,
        'eta_resolution': 0.03,
        'phi_resolution': 0.03,
    },
    'kd': {
        'enable': True,
        'temperature': 2.0,
        'alpha_kd': 0.5,
        'alpha_attn': 0,
    },
    'training': {
        'batch_size': 256,
        'epochs': 50,
        'lr': 5e-4,
        'weight_decay': 1e-5,
        'warmup_epochs': 3,
        'patience': 8,
        'early_stop_metric': 'val_auc',
        'use_sample_weight_for_all_losses': False,
        'joint_unsmear_weight': 1.6,
        'joint_cls_weight': 0.8,
        'joint_phys_weight': 0.0,
        'joint_teacher_embed_weight': 0.0,
        'feature_loss_weights': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
        'train_resmear_repeats': 3,
        'resmear_each_epoch_baselines': True,
        'resmear_each_epoch_joint': True,
        'resmear_seed_stride': 1,
    },
}

feat_names = tool.get_feat_names(CONFIG['feature_kind'])
shared_backbone_cfg = dict(CONFIG['shared_backbone'])
shared_backbone_cfg['input_dim'] = len(feat_names)
shared_backbone_cfg['max_seq_len'] = int(CONFIG['max_particles'])
CONFIG['shared_backbone'] = shared_backbone_cfg
CONFIG['tagger'] = dict(shared_backbone_cfg)
CONFIG['joint_model'] = {**shared_backbone_cfg, **dict(CONFIG['joint_model'])}

joint_init_cfg = dict(CONFIG.get('joint_init', {}))
joint_init_default_source = str(joint_init_cfg.get('load_backbone_from', 'hlt')).strip().lower()
joint_init_source_no_kd = str(joint_init_cfg.get('load_backbone_from_no_kd', joint_init_default_source)).strip().lower()
joint_init_source_with_kd = str(joint_init_cfg.get('load_backbone_from_with_kd', joint_init_default_source)).strip().lower()
for key, value in {
    'load_backbone_from_no_kd': joint_init_source_no_kd,
    'load_backbone_from_with_kd': joint_init_source_with_kd,
}.items():
    if value not in {'none', 'hlt', 'hlt+kd', 'offline'}:
        raise ValueError(f"Unsupported joint_init.{key}: {value}")
joint_init_cfg['load_backbone_from'] = joint_init_default_source
joint_init_cfg['load_backbone_from_no_kd'] = joint_init_source_no_kd
joint_init_cfg['load_backbone_from_with_kd'] = joint_init_source_with_kd
joint_init_cfg['freeze_backbone_to_z'] = bool(joint_init_cfg.get('freeze_backbone_to_z', False))
CONFIG['joint_init'] = joint_init_cfg

feature_loss_weights = np.asarray(CONFIG['training']['feature_loss_weights'], dtype=np.float32)
if feature_loss_weights.shape[0] != len(feat_names):
    raise ValueError(
        f"Expected {len(feat_names)} feature weights for {CONFIG['feature_kind']}, got {feature_loss_weights.shape[0]}"
    )
CONFIG['training']['feature_loss_weights'] = feature_loss_weights.tolist()
CONFIG['training']['use_sample_weight_for_all_losses'] = bool(
    CONFIG['training'].get('use_sample_weight_for_all_losses', True)
)

config_path = os.path.join(OUT_DIR, 'config.json')
tool.save_config(CONFIG, config_path)
print('Data path:', CONFIG['data_path'])
print('Run dir:', OUT_DIR)
print('Feature kind:', CONFIG['feature_kind'], 'feat_names:', feat_names)
print('Repeat seeds:', CONFIG['repeat_seeds'])
print('Shared backbone cfg:', CONFIG['shared_backbone'])
print('Joint init cfg:', CONFIG['joint_init'])
print('Joint init source (no_kd):', CONFIG['joint_init']['load_backbone_from_no_kd'])
print('Joint init source (with_kd):', CONFIG['joint_init']['load_backbone_from_with_kd'])
print('Joint-only cfg:', {k: v for k, v in CONFIG['joint_model'].items() if k not in CONFIG['shared_backbone']})
print('Feature loss weights:', dict(zip(feat_names, np.round(feature_loss_weights, 4))))
print('Joint physical consistency weight:', float(CONFIG['training']['joint_phys_weight']))
print('Joint teacher embed weight:', float(CONFIG['training']['joint_teacher_embed_weight']))
print('Train resmear repeats per epoch:', int(CONFIG['training'].get('train_resmear_repeats', 1)))
print('Use sample weight for all losses:', bool(CONFIG['training']['use_sample_weight_for_all_losses']))
print('Delta fusion enabled:', bool(CONFIG['joint_model']['cls_use_delta_fusion']))
print('Detach delta for classifier:', bool(CONFIG['joint_model']['cls_detach_delta_for_cls']))
print('Gate hidden dim:', int(CONFIG['joint_model']['cls_gate_hidden_dim']))
print('Gate init bias:', float(CONFIG['joint_model']['cls_gate_init_bias']))
print('Alpha init:', float(CONFIG['joint_model']['cls_alpha_init']))


In [4]:
# Load the raw constituents and build the offline / HLT views
n = int(CONFIG['n_jets'])
S = int(CONFIG['max_particles'])

with h5py.File(CONFIG['data_path'], 'r') as f:
    labels = f['labels'][:n].astype(np.int64)
    weights = f['weights'][:n].astype(np.float32)
    pt = f['fjet_clus_pt'][:n, :S].astype(np.float32)
    eta = f['fjet_clus_eta'][:n, :S].astype(np.float32)
    phi = f['fjet_clus_phi'][:n, :S].astype(np.float32)
    E = f['fjet_clus_E'][:n, :S].astype(np.float32)

constituents_raw = np.stack([pt, eta, phi, E], axis=-1)
mask_raw = pt > 0
print('Raw:', constituents_raw.shape, 'mask:', mask_raw.shape)
print('Signal:', int(labels.sum()), 'Bkg:', int((1 - labels).sum()))

hcfg = tool.HLTEffectsCfg(**CONFIG['hlt_effects'])
_, hlt_const, hlt_mask = tool.apply_hlt_effects_pair(
    constituents_raw,
    mask_raw,
    hcfg,
    seed=seed,
)

pt_thr_off = float(CONFIG['hlt_effects']['pt_threshold_offline'])
off_mask = mask_raw & (constituents_raw[:, :, 0] >= pt_thr_off)
off_const = constituents_raw.copy()
off_const[~off_mask] = 0.0
hlt_const = hlt_const.copy()
hlt_const[~hlt_mask] = 0.0

axis_off = tool.compute_jet_axis(off_const, off_mask)
axis_hlt = tool.compute_jet_axis(hlt_const, hlt_mask)
feat_off = tool.compute_features_with_axis(off_const, off_mask, axis_off, kind=CONFIG['feature_kind'])
feat_hlt = tool.compute_features_with_axis(hlt_const, hlt_mask, axis_hlt, kind=CONFIG['feature_kind'])

idx = np.arange(len(labels))
train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=seed, stratify=labels)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=seed, stratify=labels[temp_idx])
print(f'Split: train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}')

feat_means, feat_stds = tool.get_stats(feat_off, off_mask, train_idx)
feat_off_std = tool.standardize(feat_off, off_mask, feat_means, feat_stds, clip=10.0)
feat_hlt_std = tool.standardize(feat_hlt, hlt_mask, feat_means, feat_stds, clip=10.0)
common_mask = off_mask & hlt_mask

x_joint = feat_hlt_std.copy()
y_joint = feat_off_std.copy()
x_joint[~common_mask] = 0.0
y_joint[~common_mask] = 0.0

train_const_raw = constituents_raw[train_idx]
train_mask_raw = mask_raw[train_idx]

print('Offline/HLT feature shape:', feat_off_std.shape, feat_hlt_std.shape)
print('Mask identical:', bool(np.array_equal(off_mask, hlt_mask)))
print('Common-mask fraction:', float(common_mask.mean()))
print('Feat means:', np.round(feat_means, 4))
print('Feat stds :', np.round(feat_stds, 4))
print('Baseline epoch resmear enabled:', bool(CONFIG['training'].get('resmear_each_epoch_baselines', False)))
print('Joint epoch resmear enabled:', bool(CONFIG['training'].get('resmear_each_epoch_joint', False)))


Raw: (200000, 100, 4) mask: (200000, 100)
Signal: 99836 Bkg: 100164
Split: train=140,000 val=30,000 test=30,000
Offline/HLT feature shape: (200000, 100, 7) (200000, 100, 7)
Mask identical: True
Common-mask fraction: 0.5429243
Feat means: [-2.0000e-04 -1.0000e-04  8.7940e+00  9.0840e+00 -5.2585e+00 -5.2701e+00
  2.2250e-01]
Feat stds : [0.2121 0.2173 1.5182 1.5217 1.4919 1.4935 0.2067]
Baseline epoch resmear enabled: True
Joint epoch resmear enabled: True


In [ ]:
# Build the datasets and loaders
BS = int(CONFIG['training']['batch_size'])
TRAIN_RESMEAR_REPEATS = int(CONFIG['training'].get('train_resmear_repeats', 1))
if TRAIN_RESMEAR_REPEATS < 1:
    raise ValueError(f"train_resmear_repeats must be >= 1, got {TRAIN_RESMEAR_REPEATS}")

base_train_feat_off = feat_off_std[train_idx]
base_train_feat_hlt = feat_hlt_std[train_idx]
base_train_labels = labels[train_idx]
base_train_off_mask = off_mask[train_idx]
base_train_hlt_mask = hlt_mask[train_idx]
base_train_weights = weights[train_idx]
base_train_x_joint = x_joint[train_idx]
base_train_y_joint = y_joint[train_idx]
base_train_common_mask = common_mask[train_idx]

print('Train resmear repeats per epoch:', TRAIN_RESMEAR_REPEATS)
print('Base train jets:', int(base_train_labels.shape[0]))
print('Expanded training entries per epoch:', int(base_train_labels.shape[0]) * int(TRAIN_RESMEAR_REPEATS))

train_ds_hlt = tool.JetDataset(
    base_train_feat_off,
    base_train_feat_hlt,
    base_train_labels,
    base_train_off_mask,
    base_train_hlt_mask,
    base_train_weights,
)
val_ds_hlt = tool.JetDataset(
    feat_off_std[val_idx],
    feat_hlt_std[val_idx],
    labels[val_idx],
    off_mask[val_idx],
    hlt_mask[val_idx],
    weights[val_idx],
)
test_ds_hlt = tool.JetDataset(
    feat_off_std[test_idx],
    feat_hlt_std[test_idx],
    labels[test_idx],
    off_mask[test_idx],
    hlt_mask[test_idx],
    weights[test_idx],
)

train_ds_joint = tool.JointJetDataset(
    base_train_x_joint,
    base_train_y_joint,
    base_train_common_mask,
    base_train_labels,
    base_train_weights,
)

val_ds_joint = tool.JointJetDataset(
    x_joint[val_idx],
    y_joint[val_idx],
    common_mask[val_idx],
    labels[val_idx],
    weights[val_idx],
)
test_ds_joint = tool.JointJetDataset(
    x_joint[test_idx],
    y_joint[test_idx],
    common_mask[test_idx],
    labels[test_idx],
    weights[test_idx],
)

train_loader_hlt = DataLoader(train_ds_hlt, batch_size=BS, shuffle=True, drop_last=True)
val_loader_hlt = DataLoader(val_ds_hlt, batch_size=BS, shuffle=False)
test_loader_hlt = DataLoader(test_ds_hlt, batch_size=BS, shuffle=False)

train_loader_joint = DataLoader(train_ds_joint, batch_size=BS, shuffle=True, drop_last=True)
val_loader_joint = DataLoader(val_ds_joint, batch_size=BS, shuffle=False)
test_loader_joint = DataLoader(test_ds_joint, batch_size=BS, shuffle=False)


def make_epoch_hlt_train_loader(epoch: int):
    return tool.make_epoch_hlt_train_loader(
        epoch=int(epoch),
        batch_size=BS,
        feat_off_train=base_train_feat_off,
        off_mask_train=base_train_off_mask,
        labels_train=base_train_labels,
        weights_train=base_train_weights,
        train_const_raw=train_const_raw,
        train_mask_raw=train_mask_raw,
        cfg=hcfg,
        feature_kind=CONFIG['feature_kind'],
        means=feat_means,
        stds=feat_stds,
        seed=seed,
        fixed_feat_hlt_train=base_train_feat_hlt,
        fixed_hlt_mask_train=base_train_hlt_mask,
        seed_stride=int(CONFIG['training'].get('resmear_seed_stride', 1)),
        resmear_each_epoch=bool(CONFIG['training'].get('resmear_each_epoch_baselines', False)),
        clip=10.0,
        train_repeats=TRAIN_RESMEAR_REPEATS,
    )


def make_epoch_joint_train_loader(epoch: int):
    return tool.make_epoch_joint_train_loader(
        epoch=int(epoch),
        batch_size=BS,
        labels_train=base_train_labels,
        weights_train=base_train_weights,
        train_const_raw=train_const_raw,
        train_mask_raw=train_mask_raw,
        cfg=hcfg,
        feature_kind=CONFIG['feature_kind'],
        means=feat_means,
        stds=feat_stds,
        seed=seed,
        fixed_x_train=base_train_x_joint,
        fixed_y_train=base_train_y_joint,
        fixed_mask_train=base_train_common_mask,
        seed_stride=int(CONFIG['training'].get('resmear_seed_stride', 1)),
        resmear_each_epoch=bool(CONFIG['training'].get('resmear_each_epoch_joint', True)),
        clip=10.0,
        train_repeats=TRAIN_RESMEAR_REPEATS,
    )


In [ ]:
# 把所有模型按不同 seed 跑三次，并保存每次的 ckpt / metrics / predictions。
from pathlib import Path

train_cfg = CONFIG['training']
kd_cfg = CONFIG['kd']
joint_init_cfg = dict(CONFIG.get('joint_init', {}))
JOINT_INIT_SOURCE_DEFAULT = str(joint_init_cfg.get('load_backbone_from', 'hlt')).strip().lower()
JOINT_INIT_SOURCE_NO_KD = str(joint_init_cfg.get('load_backbone_from_no_kd', JOINT_INIT_SOURCE_DEFAULT)).strip().lower()
JOINT_INIT_SOURCE_WITH_KD = str(joint_init_cfg.get('load_backbone_from_with_kd', JOINT_INIT_SOURCE_DEFAULT)).strip().lower()
FREEZE_BACKBONE_TO_Z = bool(joint_init_cfg.get('freeze_backbone_to_z', False))
use_sample_weight_for_all_losses = bool(train_cfg.get('use_sample_weight_for_all_losses', True))
joint_feature_loss_weights = np.asarray(train_cfg['feature_loss_weights'], dtype=np.float32)
REPEAT_SEEDS = [int(x) for x in CONFIG.get('repeat_seeds', [42, 52, 62])]


def _set_repeat_seed(seed_value: int):
    np.random.seed(int(seed_value))
    torch.manual_seed(int(seed_value))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed_value))


def _make_repeat_hlt_train_loader(epoch: int, repeat_seed: int):
    return tool.make_epoch_hlt_train_loader(
        epoch=int(epoch),
        batch_size=BS,
        feat_off_train=base_train_feat_off,
        off_mask_train=base_train_off_mask,
        labels_train=base_train_labels,
        weights_train=base_train_weights,
        train_const_raw=train_const_raw,
        train_mask_raw=train_mask_raw,
        cfg=hcfg,
        feature_kind=CONFIG['feature_kind'],
        means=feat_means,
        stds=feat_stds,
        seed=int(repeat_seed),
        fixed_feat_hlt_train=base_train_feat_hlt,
        fixed_hlt_mask_train=base_train_hlt_mask,
        seed_stride=int(CONFIG['training'].get('resmear_seed_stride', 1)),
        resmear_each_epoch=bool(CONFIG['training'].get('resmear_each_epoch_baselines', False)),
        clip=10.0,
        train_repeats=TRAIN_RESMEAR_REPEATS,
    )


def _make_repeat_joint_train_loader(epoch: int, repeat_seed: int):
    return tool.make_epoch_joint_train_loader(
        epoch=int(epoch),
        batch_size=BS,
        labels_train=base_train_labels,
        weights_train=base_train_weights,
        train_const_raw=train_const_raw,
        train_mask_raw=train_mask_raw,
        cfg=hcfg,
        feature_kind=CONFIG['feature_kind'],
        means=feat_means,
        stds=feat_stds,
        seed=int(repeat_seed),
        fixed_x_train=base_train_x_joint,
        fixed_y_train=base_train_y_joint,
        fixed_mask_train=base_train_common_mask,
        seed_stride=int(CONFIG['training'].get('resmear_seed_stride', 1)),
        resmear_each_epoch=bool(CONFIG['training'].get('resmear_each_epoch_joint', True)),
        clip=10.0,
        train_repeats=TRAIN_RESMEAR_REPEATS,
    )


def _find_existing_repeat_dir(repeat_seed: int):
    target_suffix = f'_seed_{int(repeat_seed)}'
    if not os.path.isdir(REPEAT_DIR):
        return None
    candidates = []
    for name in os.listdir(REPEAT_DIR):
        full = os.path.join(REPEAT_DIR, name)
        if os.path.isdir(full) and str(name).endswith(target_suffix):
            candidates.append(full)
    if not candidates:
        return None
    candidates.sort()
    return candidates[0]


def _repeat_artifact_paths(repeat_dir: str):
    repeat_ckpt_dir = os.path.join(repeat_dir, 'ckpts')
    repeat_metrics_dir = os.path.join(repeat_dir, 'metrics')
    repeat_pred_dir = os.path.join(repeat_dir, 'predictions')
    return {
        'repeat_dir': repeat_dir,
        'repeat_ckpt_dir': repeat_ckpt_dir,
        'repeat_metrics_dir': repeat_metrics_dir,
        'repeat_pred_dir': repeat_pred_dir,
        'epoch_metrics_paths': {
            'teacher_off': os.path.join(repeat_metrics_dir, 'teacher_off_epoch_metrics.csv'),
            'student_hlt': os.path.join(repeat_metrics_dir, 'student_hlt_epoch_metrics.csv'),
            'hlt_kd': os.path.join(repeat_metrics_dir, 'hlt_kd_epoch_metrics.csv'),
            'joint_no_kd': os.path.join(repeat_metrics_dir, 'joint_no_kd_epoch_metrics.csv'),
            'joint_with_kd': os.path.join(repeat_metrics_dir, 'joint_with_kd_epoch_metrics.csv'),
        },
        'ckpt_paths': {
            'Teacher(OFF_FULL)': os.path.join(repeat_ckpt_dir, 'teacher_offline.pt'),
            'Student(HLT)': os.path.join(repeat_ckpt_dir, 'student_hlt.pt'),
            'Student(HLT)+KD': os.path.join(repeat_ckpt_dir, 'student_hlt_kd.pt'),
            'JointSharedEncoder(HLT,no_kd)': os.path.join(repeat_ckpt_dir, 'joint_sharedencoder_no_kd.pt'),
            'JointSharedEncoder(HLT,with_kd)': os.path.join(repeat_ckpt_dir, 'joint_sharedencoder_with_kd.pt'),
        },
        'prediction_paths': {
            'Teacher(OFF_FULL)': os.path.join(repeat_pred_dir, 'teacher_offline_test_preds.npz'),
            'Student(HLT)': os.path.join(repeat_pred_dir, 'student_hlt_test_preds.npz'),
            'Student(HLT)+KD': os.path.join(repeat_pred_dir, 'student_hlt_kd_test_preds.npz'),
            'JointSharedEncoder(HLT,no_kd)': os.path.join(repeat_pred_dir, 'joint_sharedencoder_no_kd_test_preds.npz'),
            'JointSharedEncoder(HLT,with_kd)': os.path.join(repeat_pred_dir, 'joint_sharedencoder_with_kd_test_preds.npz'),
        },
    }


def _load_prediction_bundle(path: str):
    data = np.load(path)
    return {
        'preds': np.asarray(data['preds'], dtype=np.float32),
        'labels': np.asarray(data['labels'], dtype=np.float32),
        'weights': np.asarray(data['weights'], dtype=np.float32),
    }


def _load_existing_repeat_result(repeat_idx: int, repeat_seed: int):
    repeat_dir = _find_existing_repeat_dir(int(repeat_seed))
    if repeat_dir is None:
        return None

    artifacts = _repeat_artifact_paths(repeat_dir)
    required_paths = []
    required_paths.extend(list(artifacts['ckpt_paths'].values()))
    required_paths.extend(list(artifacts['prediction_paths'].values()))
    required_paths.extend(list(artifacts['epoch_metrics_paths'].values()))
    missing_paths = [p for p in required_paths if not os.path.isfile(p)]
    if missing_paths:
        print(f'Seed {repeat_seed}: found existing folder but artifacts are incomplete, retrain this seed.')
        for p in missing_paths[:5]:
            print('  missing:', p)
        if len(missing_paths) > 5:
            print(f'  ... and {len(missing_paths) - 5} more missing files')
        return None

    model_to_metric_key = {
        'Teacher(OFF_FULL)': 'teacher_off',
        'Student(HLT)': 'student_hlt',
        'Student(HLT)+KD': 'hlt_kd',
        'JointSharedEncoder(HLT,no_kd)': 'joint_no_kd',
        'JointSharedEncoder(HLT,with_kd)': 'joint_with_kd',
    }

    models = {}
    for model_name, pred_path in artifacts['prediction_paths'].items():
        bundle = _load_prediction_bundle(pred_path)
        sample_weight = np.asarray(bundle['weights'], dtype=np.float64) if use_sample_weight_for_all_losses else None
        _fpr, _tpr, auc, auc_weighted = tool.compute_roc(bundle['labels'], bundle['preds'], sample_weight=sample_weight)
        models[model_name] = {
            'auc': float(auc),
            'auc_weighted': float(auc_weighted),
            'preds': bundle['preds'],
            'labels': bundle['labels'],
            'weights': bundle['weights'],
            'ckpt_path': artifacts['ckpt_paths'][model_name],
            'prediction_path': pred_path,
            'epoch_metrics_path': artifacts['epoch_metrics_paths'][model_to_metric_key[model_name]],
        }

    print(f'Seed {repeat_seed}: found existing repeat folder, load and skip training -> {repeat_dir}')
    return {
        'repeat_idx': int(repeat_idx),
        'repeat_seed': int(repeat_seed),
        'repeat_dir': repeat_dir,
        'models': models,
    }


repeat_results = []
print('Repeat seeds:', REPEAT_SEEDS)
print('Joint backbone init source (no_kd):', JOINT_INIT_SOURCE_NO_KD)
print('Joint backbone init source (with_kd):', JOINT_INIT_SOURCE_WITH_KD)
print('Freeze backbone to z:', FREEZE_BACKBONE_TO_Z)

for repeat_idx, repeat_seed in enumerate(REPEAT_SEEDS):
    print()
    print(f'==== Repeat {repeat_idx + 1}/{len(REPEAT_SEEDS)} seed={repeat_seed} ====')
    cached_repeat_result = _load_existing_repeat_result(int(repeat_idx), int(repeat_seed))
    if cached_repeat_result is not None:
        repeat_results.append(cached_repeat_result)
        continue

    _set_repeat_seed(int(repeat_seed))

    repeat_dir = os.path.join(REPEAT_DIR, f'repeat_{int(repeat_idx):02d}_seed_{int(repeat_seed)}')
    repeat_ckpt_dir = os.path.join(repeat_dir, 'ckpts')
    repeat_metrics_dir = os.path.join(repeat_dir, 'metrics')
    repeat_pred_dir = os.path.join(repeat_dir, 'predictions')
    tool.ensure_dir(repeat_dir)
    tool.ensure_dir(repeat_ckpt_dir)
    tool.ensure_dir(repeat_metrics_dir)
    tool.ensure_dir(repeat_pred_dir)

    epoch_metrics_paths = {
        'teacher_off': os.path.join(repeat_metrics_dir, 'teacher_off_epoch_metrics.csv'),
        'student_hlt': os.path.join(repeat_metrics_dir, 'student_hlt_epoch_metrics.csv'),
        'hlt_kd': os.path.join(repeat_metrics_dir, 'hlt_kd_epoch_metrics.csv'),
        'joint_no_kd': os.path.join(repeat_metrics_dir, 'joint_no_kd_epoch_metrics.csv'),
        'joint_with_kd': os.path.join(repeat_metrics_dir, 'joint_with_kd_epoch_metrics.csv'),
    }

    hlt_train_loader_factory = (
        (lambda epoch, repeat_seed=repeat_seed: _make_repeat_hlt_train_loader(epoch, int(repeat_seed)))
        if bool(train_cfg.get('resmear_each_epoch_baselines', False))
        else None
    )
    joint_train_loader_factory = (
        (lambda epoch, repeat_seed=repeat_seed: _make_repeat_joint_train_loader(epoch, int(repeat_seed)))
        if bool(train_cfg.get('resmear_each_epoch_joint', True))
        else None
    )

    teacher = ParticleTransformerKD(**CONFIG['tagger']).to(device)
    teacher_ckpt = os.path.join(repeat_ckpt_dir, 'teacher_offline.pt')
    teacher = tool.train_or_load_standard_model(
        'Teacher(OFF_FULL)',
        teacher,
        teacher_ckpt,
        train_loader_hlt,
        val_loader_hlt,
        device=device,
        feat_key='off',
        mask_key='mask_off',
        allow_load=bool(CONFIG.get('load_shared_baselines', False)),
        lr=float(train_cfg['lr']),
        weight_decay=float(train_cfg['weight_decay']),
        warmup_epochs=int(train_cfg['warmup_epochs']),
        epochs=int(train_cfg['epochs']),
        patience=int(train_cfg['patience']),
        early_stop_metric=str(train_cfg['early_stop_metric']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
        epoch_metrics_path=epoch_metrics_paths['teacher_off'],
    )
    auc_teacher, auc_teacher_w, p_teacher, y_true, w_true = tool.evaluate(
        teacher,
        test_loader_hlt,
        device,
        'off',
        'mask_off',
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
    )
    teacher_pred_path = os.path.join(repeat_pred_dir, 'teacher_offline_test_preds.npz')
    tool.save_prediction_bundle(teacher_pred_path, preds=p_teacher, labels=y_true, weights=w_true)

    student_hlt = ParticleTransformerKD(**CONFIG['tagger']).to(device)
    student_hlt_ckpt = os.path.join(repeat_ckpt_dir, 'student_hlt.pt')
    student_hlt = tool.train_or_load_standard_model(
        'Student(HLT)',
        student_hlt,
        student_hlt_ckpt,
        train_loader_hlt,
        val_loader_hlt,
        device=device,
        feat_key='hlt',
        mask_key='mask_hlt',
        allow_load=bool(CONFIG.get('load_shared_baselines', False)),
        lr=float(train_cfg['lr']),
        weight_decay=float(train_cfg['weight_decay']),
        warmup_epochs=int(train_cfg['warmup_epochs']),
        epochs=int(train_cfg['epochs']),
        patience=int(train_cfg['patience']),
        early_stop_metric=str(train_cfg['early_stop_metric']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
        train_loader_factory=hlt_train_loader_factory,
        epoch_metrics_path=epoch_metrics_paths['student_hlt'],
    )
    auc_hlt, auc_hlt_w, p_hlt, _, _ = tool.evaluate(
        student_hlt,
        test_loader_hlt,
        device,
        'hlt',
        'mask_hlt',
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
    )
    hlt_pred_path = os.path.join(repeat_pred_dir, 'student_hlt_test_preds.npz')
    tool.save_prediction_bundle(hlt_pred_path, preds=p_hlt, labels=y_true, weights=w_true)

    student_hlt_kd = ParticleTransformerKD(**CONFIG['tagger']).to(device)
    student_hlt_kd_ckpt = os.path.join(repeat_ckpt_dir, 'student_hlt_kd.pt')
    student_hlt_kd = tool.train_or_load_kd_standard_model(
        'Student(HLT)+KD',
        student_hlt_kd,
        teacher,
        student_hlt_kd_ckpt,
        train_loader_hlt,
        val_loader_hlt,
        device=device,
        allow_load=bool(CONFIG.get('load_shared_baselines', False)),
        lr=float(train_cfg['lr']),
        weight_decay=float(train_cfg['weight_decay']),
        warmup_epochs=int(train_cfg['warmup_epochs']),
        epochs=int(train_cfg['epochs']),
        patience=int(train_cfg['patience']),
        early_stop_metric=str(train_cfg['early_stop_metric']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
        kd_temperature=float(kd_cfg['temperature']),
        kd_alpha=float(kd_cfg['alpha_kd']),
        kd_alpha_attn=float(kd_cfg['alpha_attn']),
        train_loader_factory=hlt_train_loader_factory,
        epoch_metrics_path=epoch_metrics_paths['hlt_kd'],
    )
    hlt_kd_test = tool.eval_kd_student(
        student_hlt_kd,
        teacher,
        test_loader_hlt,
        device,
        {'kd': {'temperature': float(kd_cfg['temperature']), 'alpha_kd': float(kd_cfg['alpha_kd']), 'alpha_attn': float(kd_cfg['alpha_attn'])}},
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
    )
    hlt_kd_pred_path = os.path.join(repeat_pred_dir, 'student_hlt_kd_test_preds.npz')
    tool.save_prediction_bundle(
        hlt_kd_pred_path,
        preds=np.asarray(hlt_kd_test['preds'], dtype=np.float32),
        labels=np.asarray(hlt_kd_test['labels'], dtype=np.float32),
        weights=np.asarray(hlt_kd_test['weights'], dtype=np.float32),
    )

    joint_init_model_map = {
        'offline': teacher,
        'hlt': student_hlt,
        'hlt+kd': student_hlt_kd,
    }
    joint_init_model_no_kd = joint_init_model_map.get(JOINT_INIT_SOURCE_NO_KD)
    joint_init_model_with_kd = joint_init_model_map.get(JOINT_INIT_SOURCE_WITH_KD)

    joint_model_no_kd = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
    joint_ckpt_no_kd = os.path.join(repeat_ckpt_dir, 'joint_sharedencoder_no_kd.pt')
    if not (bool(CONFIG.get('load_joint_model', False)) and Path(joint_ckpt_no_kd).is_file()):
        if JOINT_INIT_SOURCE_NO_KD == 'none':
            print('Init JointSharedEncoder(HLT,no_kd) from random initialization (no baseline backbone load)')
        else:
            init_info = tool.initialize_joint_from_baseline(
                joint_model_no_kd,
                joint_init_model_no_kd,
                freeze_backbone_to_z=FREEZE_BACKBONE_TO_Z,
            )
            print(
                f'Init JointSharedEncoder(HLT,no_kd) from {JOINT_INIT_SOURCE_NO_KD} | '
                f'freeze_backbone_to_z={init_info["freeze_backbone_to_z"]}'
            )
    joint_model_no_kd = tool.train_or_load_joint_model(
        'JointSharedEncoder(HLT,no_kd)',
        joint_model_no_kd,
        joint_ckpt_no_kd,
        train_loader_joint,
        val_loader_joint,
        device=device,
        feat_names=feat_names,
        feat_means=feat_means,
        feat_stds=feat_stds,
        feature_loss_weights=joint_feature_loss_weights,
        joint_phys_weight=float(train_cfg['joint_phys_weight']),
        joint_unsmear_weight=float(train_cfg['joint_unsmear_weight']),
        joint_cls_weight=float(train_cfg['joint_cls_weight']),
        joint_teacher_embed_weight=float(train_cfg.get('joint_teacher_embed_weight', 0.0)),
        lr=float(train_cfg['lr']),
        weight_decay=float(train_cfg['weight_decay']),
        warmup_epochs=int(train_cfg['warmup_epochs']),
        epochs=int(train_cfg['epochs']),
        patience=int(train_cfg['patience']),
        early_stop_metric=str(train_cfg['early_stop_metric']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
        teacher=teacher,
        use_kd=False,
        kd_temperature=float(kd_cfg['temperature']),
        kd_alpha=float(kd_cfg['alpha_kd']),
        kd_alpha_attn=float(kd_cfg['alpha_attn']),
        allow_load=bool(CONFIG.get('load_joint_model', False)),
        train_loader_factory=joint_train_loader_factory,
        epoch_metrics_path=epoch_metrics_paths['joint_no_kd'],
    )
    joint_test_no_kd = tool.eval_joint_model(
        joint_model_no_kd,
        test_loader_joint,
        device=device,
        feat_names=feat_names,
        feat_means=feat_means,
        feat_stds=feat_stds,
        feature_loss_weights=joint_feature_loss_weights,
        joint_phys_weight=float(train_cfg['joint_phys_weight']),
        joint_unsmear_weight=float(train_cfg['joint_unsmear_weight']),
        joint_cls_weight=float(train_cfg['joint_cls_weight']),
        joint_teacher_embed_weight=float(train_cfg.get('joint_teacher_embed_weight', 0.0)),
        teacher=teacher,
        use_kd=False,
        kd_temperature=float(kd_cfg['temperature']),
        kd_alpha=float(kd_cfg['alpha_kd']),
        kd_alpha_attn=float(kd_cfg['alpha_attn']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
    )
    joint_no_kd_pred_path = os.path.join(repeat_pred_dir, 'joint_sharedencoder_no_kd_test_preds.npz')
    tool.save_prediction_bundle(
        joint_no_kd_pred_path,
        preds=np.asarray(joint_test_no_kd['preds'], dtype=np.float32),
        labels=np.asarray(joint_test_no_kd['labels'], dtype=np.float32),
        weights=np.asarray(joint_test_no_kd['weights'], dtype=np.float32),
    )

    joint_model_with_kd = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
    joint_ckpt_with_kd = os.path.join(repeat_ckpt_dir, 'joint_sharedencoder_with_kd.pt')
    if not (bool(CONFIG.get('load_joint_model', False)) and Path(joint_ckpt_with_kd).is_file()):
        if JOINT_INIT_SOURCE_WITH_KD == 'none':
            print('Init JointSharedEncoder(HLT,with_kd) from random initialization (no baseline backbone load)')
        else:
            init_info = tool.initialize_joint_from_baseline(
                joint_model_with_kd,
                joint_init_model_with_kd,
                freeze_backbone_to_z=FREEZE_BACKBONE_TO_Z,
            )
            print(
                f'Init JointSharedEncoder(HLT,with_kd) from {JOINT_INIT_SOURCE_WITH_KD} | '
                f'freeze_backbone_to_z={init_info["freeze_backbone_to_z"]}'
            )
    joint_model_with_kd = tool.train_or_load_joint_model(
        'JointSharedEncoder(HLT,with_kd)',
        joint_model_with_kd,
        joint_ckpt_with_kd,
        train_loader_joint,
        val_loader_joint,
        device=device,
        feat_names=feat_names,
        feat_means=feat_means,
        feat_stds=feat_stds,
        feature_loss_weights=joint_feature_loss_weights,
        joint_phys_weight=float(train_cfg['joint_phys_weight']),
        joint_unsmear_weight=float(train_cfg['joint_unsmear_weight']),
        joint_cls_weight=float(train_cfg['joint_cls_weight']),
        joint_teacher_embed_weight=float(train_cfg.get('joint_teacher_embed_weight', 0.0)),
        lr=float(train_cfg['lr']),
        weight_decay=float(train_cfg['weight_decay']),
        warmup_epochs=int(train_cfg['warmup_epochs']),
        epochs=int(train_cfg['epochs']),
        patience=int(train_cfg['patience']),
        early_stop_metric=str(train_cfg['early_stop_metric']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
        teacher=teacher,
        use_kd=True,
        kd_temperature=float(kd_cfg['temperature']),
        kd_alpha=float(kd_cfg['alpha_kd']),
        kd_alpha_attn=float(kd_cfg['alpha_attn']),
        allow_load=bool(CONFIG.get('load_joint_model', False)),
        train_loader_factory=joint_train_loader_factory,
        epoch_metrics_path=epoch_metrics_paths['joint_with_kd'],
    )
    joint_test_with_kd = tool.eval_joint_model(
        joint_model_with_kd,
        test_loader_joint,
        device=device,
        feat_names=feat_names,
        feat_means=feat_means,
        feat_stds=feat_stds,
        feature_loss_weights=joint_feature_loss_weights,
        joint_phys_weight=float(train_cfg['joint_phys_weight']),
        joint_unsmear_weight=float(train_cfg['joint_unsmear_weight']),
        joint_cls_weight=float(train_cfg['joint_cls_weight']),
        joint_teacher_embed_weight=float(train_cfg.get('joint_teacher_embed_weight', 0.0)),
        teacher=teacher,
        use_kd=True,
        kd_temperature=float(kd_cfg['temperature']),
        kd_alpha=float(kd_cfg['alpha_kd']),
        kd_alpha_attn=float(kd_cfg['alpha_attn']),
        use_sample_weight_for_all_losses=use_sample_weight_for_all_losses,
    )
    joint_with_kd_pred_path = os.path.join(repeat_pred_dir, 'joint_sharedencoder_with_kd_test_preds.npz')
    tool.save_prediction_bundle(
        joint_with_kd_pred_path,
        preds=np.asarray(joint_test_with_kd['preds'], dtype=np.float32),
        labels=np.asarray(joint_test_with_kd['labels'], dtype=np.float32),
        weights=np.asarray(joint_test_with_kd['weights'], dtype=np.float32),
    )

    repeat_results.append({
        'repeat_idx': int(repeat_idx),
        'repeat_seed': int(repeat_seed),
        'repeat_dir': repeat_dir,
        'models': {
            'Teacher(OFF_FULL)': {
                'auc': float(auc_teacher),
                'auc_weighted': float(auc_teacher_w),
                'preds': np.asarray(p_teacher, dtype=np.float32),
                'labels': np.asarray(y_true, dtype=np.float32),
                'weights': np.asarray(w_true, dtype=np.float32),
                'ckpt_path': teacher_ckpt,
                'prediction_path': teacher_pred_path,
                'epoch_metrics_path': epoch_metrics_paths['teacher_off'],
            },
            'Student(HLT)': {
                'auc': float(auc_hlt),
                'auc_weighted': float(auc_hlt_w),
                'preds': np.asarray(p_hlt, dtype=np.float32),
                'labels': np.asarray(y_true, dtype=np.float32),
                'weights': np.asarray(w_true, dtype=np.float32),
                'ckpt_path': student_hlt_ckpt,
                'prediction_path': hlt_pred_path,
                'epoch_metrics_path': epoch_metrics_paths['student_hlt'],
            },
            'Student(HLT)+KD': {
                'auc': float(hlt_kd_test['auc']),
                'auc_weighted': float(hlt_kd_test['auc_weighted']),
                'preds': np.asarray(hlt_kd_test['preds'], dtype=np.float32),
                'labels': np.asarray(hlt_kd_test['labels'], dtype=np.float32),
                'weights': np.asarray(hlt_kd_test['weights'], dtype=np.float32),
                'ckpt_path': student_hlt_kd_ckpt,
                'prediction_path': hlt_kd_pred_path,
                'epoch_metrics_path': epoch_metrics_paths['hlt_kd'],
                'test_metrics': hlt_kd_test,
            },
            'JointSharedEncoder(HLT,no_kd)': {
                'auc': float(joint_test_no_kd['auc']),
                'auc_weighted': float(joint_test_no_kd['auc_weighted']),
                'preds': np.asarray(joint_test_no_kd['preds'], dtype=np.float32),
                'labels': np.asarray(joint_test_no_kd['labels'], dtype=np.float32),
                'weights': np.asarray(joint_test_no_kd['weights'], dtype=np.float32),
                'ckpt_path': joint_ckpt_no_kd,
                'prediction_path': joint_no_kd_pred_path,
                'epoch_metrics_path': epoch_metrics_paths['joint_no_kd'],
                'test_metrics': joint_test_no_kd,
            },
            'JointSharedEncoder(HLT,with_kd)': {
                'auc': float(joint_test_with_kd['auc']),
                'auc_weighted': float(joint_test_with_kd['auc_weighted']),
                'preds': np.asarray(joint_test_with_kd['preds'], dtype=np.float32),
                'labels': np.asarray(joint_test_with_kd['labels'], dtype=np.float32),
                'weights': np.asarray(joint_test_with_kd['weights'], dtype=np.float32),
                'ckpt_path': joint_ckpt_with_kd,
                'prediction_path': joint_with_kd_pred_path,
                'epoch_metrics_path': epoch_metrics_paths['joint_with_kd'],
                'test_metrics': joint_test_with_kd,
            },
        },
    })

print()
print('Completed repeats:', len(repeat_results))


In [ ]:
# 汇总三次 repeat 的 ROC / AUC / FPR@TPR / Gap Recovery，并输出均值与方差。
import pandas as pd

use_sample_weight_for_all_losses = bool(train_cfg.get('use_sample_weight_for_all_losses', True))
MODEL_DISPLAY_ORDER = [
    'Teacher(OFF_FULL)',
    'Student(HLT)',
    'Student(HLT)+KD',
    'JointSharedEncoder(HLT,no_kd)',
    'JointSharedEncoder(HLT,with_kd)',
]
MODEL_COLORS = {
    'Teacher(OFF_FULL)': '#4C78A8',
    'Student(HLT)': '#F58518',
    'Student(HLT)+KD': '#54A24B',
    'JointSharedEncoder(HLT,no_kd)': '#E45756',
    'JointSharedEncoder(HLT,with_kd)': '#72B7B2',
}


def _gap_recovery(model_fpr: float, baseline_fpr: float, teacher_fpr: float) -> float:
    denom = float(baseline_fpr - teacher_fpr)
    if abs(denom) < 1e-12:
        return float('nan')
    return float((baseline_fpr - model_fpr) / denom)


repeat_detail_rows = []
for repeat_result in repeat_results:
    repeat_seed = int(repeat_result['repeat_seed'])
    teacher_result = repeat_result['models']['Teacher(OFF_FULL)']
    hlt_result = repeat_result['models']['Student(HLT)']
    teacher_weight = np.asarray(teacher_result['weights'], dtype=np.float64) if use_sample_weight_for_all_losses else None
    hlt_weight = np.asarray(hlt_result['weights'], dtype=np.float64) if use_sample_weight_for_all_losses else None
    teacher_fpr, teacher_tpr, _, _ = tool.compute_roc(teacher_result['labels'], teacher_result['preds'], sample_weight=teacher_weight)
    hlt_fpr, hlt_tpr, _, _ = tool.compute_roc(hlt_result['labels'], hlt_result['preds'], sample_weight=hlt_weight)
    teacher_fpr_30 = tool.fpr_at_target_tpr(teacher_tpr, teacher_fpr, 0.30)
    teacher_fpr_50 = tool.fpr_at_target_tpr(teacher_tpr, teacher_fpr, 0.50)
    hlt_fpr_30 = tool.fpr_at_target_tpr(hlt_tpr, hlt_fpr, 0.30)
    hlt_fpr_50 = tool.fpr_at_target_tpr(hlt_tpr, hlt_fpr, 0.50)

    for model_name in MODEL_DISPLAY_ORDER:
        model_result = repeat_result['models'][model_name]
        sample_weight = np.asarray(model_result['weights'], dtype=np.float64) if use_sample_weight_for_all_losses else None
        fpr, tpr, auc_raw, auc_weighted_raw = tool.compute_roc(model_result['labels'], model_result['preds'], sample_weight=sample_weight)
        fpr_30 = tool.fpr_at_target_tpr(tpr, fpr, 0.30)
        fpr_50 = tool.fpr_at_target_tpr(tpr, fpr, 0.50)
        repeat_detail_rows.append({
            'repeat_seed': int(repeat_seed),
            'model': str(model_name),
            'auc': float(model_result.get('auc', auc_raw)),
            'auc_weighted': float(model_result.get('auc_weighted', auc_weighted_raw)),
            'fpr_at_tpr30': float(fpr_30),
            'fpr_at_tpr50': float(fpr_50),
            'gap_recovery_tpr30': float(_gap_recovery(fpr_30, hlt_fpr_30, teacher_fpr_30)),
            'gap_recovery_tpr50': float(_gap_recovery(fpr_50, hlt_fpr_50, teacher_fpr_50)),
            'ckpt_path': str(model_result.get('ckpt_path', '')),
            'prediction_path': str(model_result.get('prediction_path', '')),
            'epoch_metrics_path': str(model_result.get('epoch_metrics_path', '')),
        })

repeat_detail_df = pd.DataFrame(repeat_detail_rows)
summary_rows = []
for model_name in MODEL_DISPLAY_ORDER:
    df_model = repeat_detail_df[repeat_detail_df['model'] == str(model_name)].copy()
    if df_model.empty:
        continue
    summary_rows.append({
        'model': str(model_name),
        'auc_mean': float(df_model['auc'].mean()),
        'auc_std': float(df_model['auc'].std(ddof=0)),
        'auc_var': float(df_model['auc'].var(ddof=0)),
        'auc_weighted_mean': float(df_model['auc_weighted'].mean()),
        'auc_weighted_std': float(df_model['auc_weighted'].std(ddof=0)),
        'auc_weighted_var': float(df_model['auc_weighted'].var(ddof=0)),
        'fpr_at_tpr30_mean': float(df_model['fpr_at_tpr30'].mean()),
        'fpr_at_tpr30_std': float(df_model['fpr_at_tpr30'].std(ddof=0)),
        'fpr_at_tpr30_var': float(df_model['fpr_at_tpr30'].var(ddof=0)),
        'fpr_at_tpr50_mean': float(df_model['fpr_at_tpr50'].mean()),
        'fpr_at_tpr50_std': float(df_model['fpr_at_tpr50'].std(ddof=0)),
        'fpr_at_tpr50_var': float(df_model['fpr_at_tpr50'].var(ddof=0)),
        'gap_recovery_tpr30_mean': float(df_model['gap_recovery_tpr30'].mean()),
        'gap_recovery_tpr30_std': float(df_model['gap_recovery_tpr30'].std(ddof=0)),
        'gap_recovery_tpr30_var': float(df_model['gap_recovery_tpr30'].var(ddof=0)),
        'gap_recovery_tpr50_mean': float(df_model['gap_recovery_tpr50'].mean()),
        'gap_recovery_tpr50_std': float(df_model['gap_recovery_tpr50'].std(ddof=0)),
        'gap_recovery_tpr50_var': float(df_model['gap_recovery_tpr50'].var(ddof=0)),
    })
summary_df = pd.DataFrame(summary_rows)

repeat_detail_out = os.path.join(TABLE_DIR, 'joint_repeat_detail.csv')
repeat_summary_out = os.path.join(TABLE_DIR, 'joint_repeat_summary.csv')
tool.save_rows_csv(repeat_detail_out, repeat_detail_rows)
tool.save_rows_csv(repeat_summary_out, summary_rows)
print('Saved table:', repeat_detail_out)
print('Saved table:', repeat_summary_out)

common_tpr = np.linspace(0.0, 1.0, 401)
plt.figure(figsize=(7.4, 6.2))
for model_name in MODEL_DISPLAY_ORDER:
    fpr_rows = []
    auc_rows = []
    auc_weighted_rows = []
    for repeat_result in repeat_results:
        model_result = repeat_result['models'][model_name]
        sample_weight = np.asarray(model_result['weights'], dtype=np.float64) if use_sample_weight_for_all_losses else None
        fpr, tpr, auc_raw, auc_weighted_raw = tool.compute_roc(model_result['labels'], model_result['preds'], sample_weight=sample_weight)
        fpr_rows.append(np.interp(common_tpr, tpr, fpr))
        auc_rows.append(float(model_result.get('auc', auc_raw)))
        auc_weighted_rows.append(float(model_result.get('auc_weighted', auc_weighted_raw)))
    fpr_arr = np.asarray(fpr_rows, dtype=np.float64)
    mean_fpr = fpr_arr.mean(axis=0)
    std_fpr = fpr_arr.std(axis=0)
    color = MODEL_COLORS.get(model_name, None)
    label = (
        f"{model_name} AUC={np.mean(auc_rows):.4f}±{np.std(auc_rows):.4f}, "
        f"wAUC={np.mean(auc_weighted_rows):.4f}±{np.std(auc_weighted_rows):.4f}"
    )
    plt.semilogy(common_tpr, np.clip(mean_fpr, 1e-6, 1.0), lw=2.0, color=color, label=label)
    plt.fill_between(
        common_tpr,
        np.clip(mean_fpr - std_fpr, 1e-6, 1.0),
        np.clip(mean_fpr + std_fpr, 1e-6, 1.0),
        color=color,
        alpha=0.12,
    )
plt.xlabel('True Positive Rate (Signal efficiency)')
plt.ylabel('False Positive Rate')
plt.title('Shared-encoder joint training ROC mean±std (test)')
plt.xlim(0.0, 1.0)
plt.ylim(1e-4, 1.0)
plt.grid(True, which='both', alpha=0.3)
plt.legend(loc='upper left', fontsize=8)
plt.tight_layout()
roc_out = os.path.join(FIG_DIR, 'sharedencoder_joint_downstream_roc_mean_std.png')
plt.savefig(roc_out, dpi=180, bbox_inches='tight')
print('Saved figure:', roc_out)
plt.show()

joint_only = repeat_detail_df[repeat_detail_df['model'].isin(['JointSharedEncoder(HLT,no_kd)', 'JointSharedEncoder(HLT,with_kd)'])].copy()
BEST_JOINT_ROW = joint_only.sort_values(['auc', 'auc_weighted'], ascending=False).iloc[0].to_dict()
print('Best joint model by test auc:', BEST_JOINT_ROW['model'], 'repeat_seed=', int(BEST_JOINT_ROW['repeat_seed']))
print('Best joint auc:', float(BEST_JOINT_ROW['auc']))
print('Best joint weighted auc:', float(BEST_JOINT_ROW['auc_weighted']))

summary_display_df = summary_df.copy()
for col in summary_display_df.columns:
    if col == 'model':
        continue
    if col.startswith('gap_recovery') or col.startswith('fpr_at_'):
        summary_display_df[col] = summary_display_df[col].map(lambda x: 'nan' if pd.isna(x) else f'{100.0 * float(x):.4f}%')
    else:
        summary_display_df[col] = summary_display_df[col].map(lambda x: f'{float(x):.6f}')

try:
    from IPython.display import display
    display(repeat_detail_df)
    display(summary_display_df)
except Exception:
    print(repeat_detail_df.to_string(index=False))
    print(summary_display_df.to_string(index=False))


In [ ]:
# 用测试 AUC 最高的 joint 模型查看特征重建表现。
import pandas as pd


@torch.no_grad()
def predict_joint_reco(model, loader):
    model.eval()
    outs = []
    for batch in loader:
        x = batch['x'].to(device)
        m = batch['mask'].to(device)
        reco, _logits = model(x, m)
        outs.append(reco.detach().cpu().numpy())
    return np.concatenate(outs, axis=0)


def metric_dict(res_1d: np.ndarray):
    if res_1d.size == 0:
        return {
            'bias': np.nan,
            'mae': np.nan,
            'rmse': np.nan,
            'abs_p50': np.nan,
            'abs_p90': np.nan,
            'abs_p99': np.nan,
        }
    abs_r = np.abs(res_1d)
    return {
        'bias': float(np.mean(res_1d)),
        'mae': float(np.mean(abs_r)),
        'rmse': float(np.sqrt(np.mean(res_1d ** 2))),
        'abs_p50': float(np.quantile(abs_r, 0.50)),
        'abs_p90': float(np.quantile(abs_r, 0.90)),
        'abs_p99': float(np.quantile(abs_r, 0.99)),
    }


def maybe_wrap_residual(name: str, feat_idx: int, residual: np.ndarray) -> np.ndarray:
    if name == 'dPhi':
        sc = float(feat_stds[feat_idx])
        return tool.wrap_dphi_np(residual * sc) / sc
    return residual


best_joint_model_name = str(BEST_JOINT_ROW['model'])
best_joint_repeat_seed = int(BEST_JOINT_ROW['repeat_seed'])
best_joint_ckpt = str(BEST_JOINT_ROW['ckpt_path'])

best_joint_model = SharedEncoderUnsmearClassifier(**CONFIG['joint_model']).to(device)
tool.load_checkpoint(best_joint_model, best_joint_ckpt, map_location=device)
best_joint_model.eval()

pred_best_joint_reco = predict_joint_reco(best_joint_model, test_loader_joint)
x_test_std = x_joint[test_idx]
y_test_std = y_joint[test_idx]
mask_test = common_mask[test_idx]

residual_sources = {
    'hlt': x_test_std - y_test_std,
    'best_joint': pred_best_joint_reco - y_test_std,
}
plot_labels = {
    'hlt': 'HLT baseline (post - pre)',
    'best_joint': f"{best_joint_model_name} | seed={best_joint_repeat_seed}",
}
plot_colors = {
    'hlt': '#4C78A8',
    'best_joint': '#54A24B' if 'with_kd' in best_joint_model_name else '#F58518',
}

metrics_rows = []
for feat_idx, feat_name in enumerate(feat_names):
    plt.figure(figsize=(6.6, 4.6))
    for method_name in ['hlt', 'best_joint']:
        residual = residual_sources[method_name][..., feat_idx][mask_test]
        residual = maybe_wrap_residual(feat_name, feat_idx, residual)
        plt.hist(
            residual,
            bins=120,
            density=True,
            alpha=0.35,
            label=plot_labels[method_name],
            color=plot_colors[method_name],
        )
        mm = metric_dict(residual)
        metrics_rows.append({
            'feature': feat_name,
            'method': method_name,
            'best_model': best_joint_model_name if method_name == 'best_joint' else 'hlt',
            'best_repeat_seed': best_joint_repeat_seed if method_name == 'best_joint' else np.nan,
            **mm,
        })

    plt.title(f'Residual compare: {feat_name}')
    plt.xlabel('Residual (std space)')
    plt.ylabel('Density')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    out = os.path.join(FIG_DIR, f'joint_reco_residual_compare_best_{feat_name}.png')
    plt.savefig(out, dpi=160, bbox_inches='tight')
    print('Saved figure:', out)
    plt.show()

metrics_df = pd.DataFrame(metrics_rows)
metrics_df = metrics_df[['feature', 'method', 'best_model', 'best_repeat_seed', 'bias', 'mae', 'rmse', 'abs_p50', 'abs_p90', 'abs_p99']]

print()
print('=' * 120)
print(f'Metrics summary (std space) | split=test | best_joint={best_joint_model_name} | repeat_seed={best_joint_repeat_seed}')
print('=' * 120)
print(metrics_df.to_string(index=False, float_format=lambda x: f'{x:.6f}'))

metrics_out = os.path.join(TABLE_DIR, 'joint_reco_metrics_summary_best_auc_model_test.csv')
metrics_df.to_csv(metrics_out, index=False)
print('Saved table:', metrics_out)

try:
    from IPython.display import display
    display(metrics_df)
except Exception:
    pass


In [ ]:
# 对三次 repeat 分别画 gate / alpha diagnostics。
import pandas as pd

plot_colors = {
    'joint_no_kd': '#F58518',
    'joint_with_kd': '#54A24B',
}

for repeat_result in repeat_results:
    repeat_seed = int(repeat_result['repeat_seed'])
    joint_metric_paths = {
        'joint_no_kd': str(repeat_result['models']['JointSharedEncoder(HLT,no_kd)']['epoch_metrics_path']),
        'joint_with_kd': str(repeat_result['models']['JointSharedEncoder(HLT,with_kd)']['epoch_metrics_path']),
    }

    joint_metric_frames = {}
    for model_name, metric_path in joint_metric_paths.items():
        if not os.path.isfile(metric_path):
            print(f'Missing epoch metrics for {model_name}:', metric_path)
            continue
        df = pd.read_csv(metric_path)
        if 'epoch' not in df.columns:
            print(f'Skip {model_name}: missing epoch column in', metric_path)
            continue
        joint_metric_frames[model_name] = df.sort_values('epoch').reset_index(drop=True)

    if not joint_metric_frames:
        print(f'Skip gate diagnostics for repeat seed={repeat_seed}: no metric tables found.')
        continue

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
    for model_name, df in joint_metric_frames.items():
        x = df['epoch'].to_numpy()
        color = plot_colors.get(model_name, None)

        if {'train_gate_mean', 'train_gate_std'}.issubset(df.columns):
            y = df['train_gate_mean'].to_numpy(dtype=float)
            y_std = df['train_gate_std'].to_numpy(dtype=float)
            axes[0].plot(x, y, marker='o', ms=4, lw=2.0, color=color, label=f'{model_name} train mean')
            axes[0].fill_between(x, np.clip(y - y_std, 0.0, 1.0), np.clip(y + y_std, 0.0, 1.0), color=color, alpha=0.16, label=f'{model_name} train std')

        if {'val_gate_mean', 'val_gate_std'}.issubset(df.columns):
            y = df['val_gate_mean'].to_numpy(dtype=float)
            y_std = df['val_gate_std'].to_numpy(dtype=float)
            axes[0].plot(x, y, marker='s', ms=4, lw=1.8, ls='--', color=color, alpha=0.95, label=f'{model_name} val mean')
            axes[0].fill_between(x, np.clip(y - y_std, 0.0, 1.0), np.clip(y + y_std, 0.0, 1.0), color=color, alpha=0.08, label=f'{model_name} val std')

        if 'alpha' in df.columns:
            axes[1].plot(x, df['alpha'], marker='o', ms=4, lw=2.0, color=color, label=model_name)

    axes[0].set_title(f'Gate mean with std band | seed={repeat_seed}')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Gate value')
    axes[0].set_ylim(0.0, 1.0)
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(fontsize=8)

    axes[1].set_title(f'Alpha across epochs | seed={repeat_seed}')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Alpha')
    axes[1].grid(True, alpha=0.25)
    axes[1].legend(fontsize=8)

    fig.suptitle(f'Delta-fusion gate diagnostics | repeat seed={repeat_seed}')
    fig.tight_layout()

    gate_diag_out = os.path.join(FIG_DIR, f'joint_gate_alpha_over_epochs_seed_{repeat_seed}.png')
    plt.savefig(gate_diag_out, dpi=170, bbox_inches='tight')
    print('Saved figure:', gate_diag_out)
    plt.show()
